<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 2 — Su primer motor de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">TF-IDF + similitud coseno, implementados desde cero · sin librerías de NLP</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Dado el corpus que preprocesaron en el **Lab 1**, responder consultas en lenguaje libre devolviendo un **ranking por relevancia**. La regla del curso aplica: los algoritmos núcleo (TF, IDF, TF-IDF y coseno) se implementan **desde cero**. Está prohibido `TfidfVectorizer` u otra caja negra para resolver el ejercicio (solo se permite, opcionalmente, al final para *verificar*).


## 0 · Cargar el corpus procesado del Lab 1

Debe estar `corpus_procesado.json` en la misma carpeta. Si no, vuelvan a correr el Lab 1.

In [ ]:
import json, math

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]          # lista de listas de terminos
print(f'{len(corpus)} documentos.  Ejemplo {corpus[0]["id"]}:', documentos[0][:8])

**Reutilicen su `preprocesar`.** Péguenla aquí *idéntica* a la del Lab 1. Es indispensable que la consulta pase por **exactamente el mismo pipeline** que el corpus (este es el error más común: vectorizar la consulta distinto y que nada coincida).

In [ ]:
def preprocesar(texto):
    # TODO: peguen aqui su funcion preprocesar() del Lab 1, sin cambios.
    raise NotImplementedError

## 1 · Indexar — TF, IDF y TF-IDF desde cero

Implementen las tres funciones siguiendo las fórmulas de la clase. La estructura es la del slide de "15 líneas".
- `tf(doc)` → frecuencia **relativa**: `f(t,d) / |d|` (importancia local).
- `idf(corpus)` → `ln(N / df(t))` (rareza global). Cuenten **documentos**, no apariciones.
- `tfidf(doc, idf_)` → `tf(t,d) · idf(t)`.


In [ ]:
def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    # TODO
    raise NotImplementedError

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    # TODO: df(t) = en cuantos documentos aparece t (usar set por documento)
    raise NotImplementedError

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    # TODO: terminos no vistos en idf_ -> peso 0
    raise NotImplementedError

In [ ]:
# Construir el indice: un vector tf-idf (dict) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check: el termino mas pesado de d04 (sequia/maiz) deberia ser tematico
import operator
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Terminos top de', corpus[3]['id'], '->', top)

## 2 · Procesar la consulta

La consulta es texto libre. Pásenla por el **mismo** `preprocesar`, conviértanla en vector con el **mismo** `IDF` del corpus (no recalculen IDF con la consulta dentro).

In [ ]:
def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    # TODO: preprocesar -> tf -> multiplicar por IDF del corpus
    raise NotImplementedError

# print(vectorizar_consulta('sequia en los cultivos'))

## 3 · Ranquear — similitud coseno

Implementen el coseno entre dos vectores dispersos (dicts) y la función `buscar` que devuelve el **top-k**.
$$\cos(\vec{q},\vec{d}) = \frac{\vec{q}\cdot\vec{d}}{\lVert\vec{q}\rVert\,\lVert\vec{d}\rVert}$$

In [ ]:
def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # TODO
    raise NotImplementedError

def buscar(consulta, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)
    # TODO: coseno de q contra cada documento del INDICE, ordenar desc, devolver top-k
    raise NotImplementedError

In [ ]:
# Prueba: deberia rankear alto d02 (sequia/maiz) y d04
for id_, titulo, score in buscar('sequia y cultivos de maiz'):
    print(f'{score:.3f}  {id_}  {titulo}')

## 4 · Rómpanlo

Entender dónde falla un modelo vale más que celebrarlo donde funciona. Empezamos con el caso de clase:


In [ ]:
print('Consulta: "problemas de agua"')
for id_, titulo, score in buscar('problemas de agua'):
    print(f'{score:.3f}  {id_}  {titulo}')

# Observen: d13 (agua potable) sale bien, pero d02 (crisis HIDRICA) — claramente relevante —
# queda hundido o en 0. 'agua' e 'hidrico' son cadenas distintas: TF-IDF no sabe que son sinonimos.

**4.a** Encuentren **2 consultas propias** donde su buscador devuelva resultados malos. Para cada una: muestren la salida y **expliquen la causa** (sinonimia, polisemia, falta de contexto, preprocesamiento agresivo, etc.). Traigan estas dos consultas a la próxima clase: con ellas abriremos BM25.

In [ ]:
# TODO: consulta fallida 1
raise NotImplementedError

_Causa de la falla 1:_ 

In [ ]:
# TODO: consulta fallida 2
raise NotImplementedError

_Causa de la falla 2:_ 

## 5 · (Opcional) Verificación contra scikit-learn

Solo para **comprobar** que su implementación desde cero es correcta — no sustituye al ejercicio ni cuenta como entrega. Si sus rankings coinciden a grandes rasgos con los de `TfidfVectorizer`, van bien.

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# docs_txt = [' '.join(t) for t in documentos]
# vec = TfidfVectorizer()
# X = vec.fit_transform(docs_txt)
# q = vec.transform([' '.join(preprocesar('sequia y cultivos de maiz'))])
# sims = cosine_similarity(q, X)[0]
# print(sorted(zip(sims, [d['id'] for d in corpus]), reverse=True)[:5])

## Entregables del Lab 2

- [ ] `tf`, `idf`, `tfidf`, `coseno`, `vectorizar_consulta` y `buscar` implementadas desde cero.
- [ ] El índice construido y la prueba de `buscar` ejecutada con salida.
- [ ] La demostración de la falla "problemas de agua" vs. "crisis hídrica" ejecutada.
- [ ] **2 consultas fallidas propias** con su causa explicada por escrito.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
